In [ ]:
from transformers import GPT2LMHeadModel
import torch

model_hugging_face = GPT2LMHeadModel.from_pretrained('gpt2') # 124M Parameters
model = model_hugging_face.state_dict()

torchinfo can some what be used? It's not accuratly displaying though, so we just use this iteration method

https://stackoverflow.com/questions/68577198/pytorch-summary-fails-with-huggingface-model

In [ ]:
for param_name, param in model.items():
    print(param_name, param.shape)

transformer.wte.weight torch.Size([50257, 768])
transformer.wpe.weight torch.Size([1024, 768])
transformer.h.0.ln_1.weight torch.Size([768])
transformer.h.0.ln_1.bias torch.Size([768])
transformer.h.0.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.0.attn.c_attn.bias torch.Size([2304])
transformer.h.0.attn.c_proj.weight torch.Size([768, 768])
transformer.h.0.attn.c_proj.bias torch.Size([768])
transformer.h.0.ln_2.weight torch.Size([768])
transformer.h.0.ln_2.bias torch.Size([768])
transformer.h.0.mlp.c_fc.weight torch.Size([768, 3072])
transformer.h.0.mlp.c_fc.bias torch.Size([3072])
transformer.h.0.mlp.c_proj.weight torch.Size([3072, 768])
transformer.h.0.mlp.c_proj.bias torch.Size([768])
transformer.h.1.ln_1.weight torch.Size([768])
transformer.h.1.ln_1.bias torch.Size([768])
transformer.h.1.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.1.attn.c_attn.bias torch.Size([2304])
transformer.h.1.attn.c_proj.weight torch.Size([768, 768])
transformer.h.1.attn.c_proj.bias 

As we can see, for GPT2:

*   vocab_size = 50257
*   embedding_size/model_size = 768
*   positional_encoding = 1024
*   num_layers = 12

# Importing Hugging Face GPT2

problem is, the way I named the variables and stuff aren't the same as how huggingface's gpt2 is

so I cant just load the tensors over, so we'll just copy the code from andrew and get the hugging face model from there

https://github.com/karpathy/build-nanogpt/blob/master/train_gpt2.py

In [ ]:
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    block_size: int = 1024 # max sequence length
    vocab_size: int = 50257 # number of tokens: 50,000 BPE merges + 256 bytes tokens + 1 <|endoftext|> token
    n_layer: int = 12 # number of layers
    n_head: int = 12 # number of heads
    n_embd: int = 768 # embedding dimension


class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)
        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # nh is "number of heads", hs is "head size", and C (number of channels) = nh * hs
        # e.g. in GPT-2 (124M), n_head=12, hs=64, so nh*hs=C=768 channels in the Transformer
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True) # flash attention
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side
        # output projection
        y = self.c_proj(y)
        return y


class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu    = nn.GELU(approximate='tanh')
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x


class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # weight sharing scheme
        self.transformer.wte.weight = self.lm_head.weight

        # init params
        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'NANOGPT_SCALE_INIT'):
                std *= (2 * self.config.n_layer) ** -0.5
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):

        # idx is of shape (B, T)
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length {T}, block size is only {self.config.block_size}"

        # forward the token and posisition embeddings
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device) # shape (T)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (T, n_embd)
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (B, T, n_embd)
        x = tok_emb + pos_emb

        # forward the blocks of the transformer
        for block in self.transformer.h:
            x = block(x)

        # forward the final layernorm and the classifier
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)

        # loss = None
        # if targets is not None:
        #     loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        # return logits, loss

        return logits

    @classmethod
    def from_pretrained(cls, model_type):
        """Loads pretrained GPT-2 model weights from huggingface"""
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

# Generating Some Text

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device=device)

model = GPT.from_pretrained('gpt2')
print("model loaded")
model.eval()
model.to(device);

loading weights from pretrained gpt: gpt2


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

model loaded


In [ ]:
!pip install tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.7 MB/s eta 0:00:00


In [ ]:
import tiktoken

num_return_sequences = 5
max_length = 512

tokenizer = tiktoken.get_encoding("gpt2")
tokens = tokenizer.encode("Hi, it's been a year since I have met you. I think you are an interesting advancement in the field of AI, but I think the capabilities of this model architecture will come to an end")
tokens = torch.tensor(tokens, dtype=torch.long) # (40)

# the reason that we are "repeating" these tokens 5 times is we want to see how GPT2 responds differently each time to the same inputs
tokens = tokens.unsqueeze(0).repeat(num_return_sequences, 1) # (5, 40)
tokens = tokens.to(device)


# setting seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed(42)


# generate
while tokens.size(1) < max_length:

    with torch.no_grad():
        logits = model(tokens) # (B, T, vocab_size)

        # take the logits of last position
        logits = logits[:, -1, :] # (B, vocab_size)

        # turn into probabilities
        probs = torch.softmax(logits, dim=-1) # (B, vocab_size)

        # only keep the top 50 probabilities
        topk_probs, topk_indices = torch.topk(probs, 50, dim=-1) # (B, 50)

        # select a token in topk
        topk_token_index = torch.multinomial(topk_probs, num_samples=1) # (B, 1)

        # gather the correspoding index in the vocabulary
        next_token_index = torch.gather(topk_indices, dim=-1, index=topk_token_index) # (B, 1)

        # there should be a if statement here to break when we meet <|endoftext|>
        # but since it's done in batches, if one of them has <|endoftext|> I don't want the others to terminate! So no if condition here

        # append to the tokens
        tokens = torch.cat((tokens, next_token_index), dim=1) # (B, T+1)

In [ ]:
# print generated text
for index in range(num_return_sequences):

    text = tokens[index, :max_length].tolist()
    decoded = tokenizer.decode(text)

    print("<New Conversation>")
    print("-----------------")
    print(decoded)
    print("-----------------")
    for _ in range(3):
        print()

<New Conversation>
-----------------
Hi, it's been a year since I have met you. I think you are an interesting advancement in the field of AI, but I think the capabilities of this model architecture will come to an end in the years to come. The current limitations of the development model make it extremely difficult for scientists to make intelligent decisions and so I am sad to see our science community have to go through what I am sure would be a very hard reevaluation of the model architecture."

So does the technology stack make sense for the future of scientific collaboration?

"As with any industry, many scientists and others who have made contributions to the field of AI make their own decisions because that's where they grow and change. And as a profession we need AI to take action, so that we do not run into the same problems at work as those at home, or at home that could jeopardize our research efforts. Even as AI can evolve, it is necessary for scientists to look at the com

You'll realize this feels far off from a "Chattable GPT"

And that is because that requires more fine tuning on special Q&A datasets

Which isn't our main purpose at the moment

# Implementing GPT2's Architecture

GPT2 still has a few changes from GPT1:

*   normal distribution initialization for linear layers
*   parameter sharing for token embedding and projection

In [ ]:
from dataclasses import dataclass
from math import sqrt
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class Our_GPT2Config:
    vocab_size: int = 50257 # number of tokens: 50,000 BPE merges + 256 bytes tokens + 1 <|endoftext|> token
    embedding_size: int = 768 # embedding dimension
    seq_len: int = 1024 # max sequence length
    num_layers: int = 12 # number of layers
    num_heads: int = 12 # number of heads



# Exact Same Functionality as Char_GPT's Implementation of Attention, Just More Optimized!
class CasualSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.key = nn.Linear(config.embedding_size, config.embedding_size, bias=False)
        self.query = nn.Linear(config.embedding_size, config.embedding_size, bias=False)
        self.value = nn.Linear(config.embedding_size, config.embedding_size, bias=False)
        self.project = nn.Linear(config.embedding_size, config.embedding_size)
        self.project.flag = 1 # a "flag" for model initialization, feels like pytorch should have better implementation?

        self.num_heads = config.num_heads
        self.head_size = config.embedding_size // config.num_heads
        self.embedding_size = config.embedding_size
        self.register_buffer("mask_indexes", torch.tril(torch.ones(config.seq_len, config.seq_len)).view(1, 1, config.seq_len, config.seq_len))

    def forward(self, x):
        B, T, C = x.size() # batch_size, seq_len, embedding_size (which sometimes is called model_size)

        # Get QKV
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        # Divide into heads (man I don't know how to reshape tensors)
        k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2) # (B, num_heads, T, head_size)
        q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2) # (B, num_heads, T, head_size)
        v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2) # (B, num_heads, T, head_size)

        # Attention
        attn = (q @ k.transpose(-2, -1)) * (1.0/sqrt(self.head_size)) # normalize by dividng by head_size
        attn = attn.masked_fill(self.mask_indexes[:,:,:T,:T] == 0, float('-inf'))
        attn = torch.softmax(attn, dim=-1)
        out = attn @ v  # (B, num_heads, T, T) @ (B, num_heads, T, head_size) -> (B, num_heads, T, head_size)

        # Get all the head outputs together
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        # Projection
        out = self.project(out)

        return out



class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.mlp_in = nn.Linear(config.embedding_size, 4 * config.embedding_size)
        self.GELU = nn.GELU()
        self.mlp_out = nn.Linear(4 * config.embedding_size, config.embedding_size)
        self.mlp_out.flag = 1 # a "flag" for model initialization, feels like pytorch should have better implementation?

    def forward(self, x):
        x = self.mlp_in(x)
        x = self.GELU(x)
        x = self.mlp_out(x)
        return x


class GPT_Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(config.embedding_size)
        self.attn = CasualSelfAttention(config)
        self.layer_norm_2 = nn.LayerNorm(config.embedding_size)
        self.mlp = MLP(config)

    # norm & add
    def forward(self, x):
        x = x + self.attn(self.layer_norm_1(x))
        x = x + self.mlp(self.layer_norm_2(x))
        return x



class Our_GPT2(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            token_embeddings = nn.Embedding(config.vocab_size, config.embedding_size),
            positional_encodings = nn.Embedding(config.seq_len, config.embedding_size),
            blocks = nn.ModuleList([GPT_Block(config) for _ in range(config.num_layers)]),
            layer_norm_final = nn.LayerNorm(config.embedding_size),
            projection = nn.Linear(config.embedding_size, config.vocab_size)
        ))

        # weight sharing scheme, so these two "share" the same tensor, and apparently this just works better than them having separate values?
        self.transformer.token_embeddings.weight = self.transformer.projection.weight

        # init weights
        self.apply(self.init_weights)


    # initialize the weights of linear layers to a normal distribution
    def init_weights(self, module):

        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'flag'):
                std *= 1/sqrt(self.config.num_layers)
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


    def forward(self, inputs):

        # inputs is of shape (B, T)
        B, T = inputs.size()

        # forward the token and position embeddings
        tok_emb = self.transformer.token_embeddings(inputs) # (B,T,C)
        pos_emb = self.transformer.positional_encodings(torch.arange(inputs.size(1), device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)

        # forward the blocks of the transformer
        for block in self.transformer.blocks:
            x = block(x)

        # forward the final layernorm and the classifier
        x = self.transformer.layer_norm_final(x)
        logits = self.transformer.projection(x) # (B, T, vocab_size)

        return logits


    def generate(self, tokens, max_seq_len):

        # inputs is (B, T) array of indices in the current context
        # but what about the conditional of '<|endoftext|>' token...?
        while tokens.size(1) <  max_seq_len:

            with torch.no_grad():

                # crop idx to the last seq_len tokens
                # context_window_tokens = tokens[:, -seq_len:]

                # get the predictions
                logits = model(tokens) # (B, T, vocab_size)

                # focus only on the last time step
                logits = logits[:, -1, :] # (B, T, vocab_size) becomes (B, vocab_size)

                # apply softmax to get probabilities
                probs = F.softmax(logits, dim=-1) # (B, vocab_size)

                # only keep the top 50 probabilities
                topk_probs, topk_indices = torch.topk(probs, 50, dim=-1) # (B, 50)

                # select a token in topk
                topk_token_index = torch.multinomial(topk_probs, num_samples=1) # (B, 1)

                # gather the correspoding index in the vocabulary
                next_token_index = torch.gather(topk_indices, dim=-1, index=topk_token_index) # (B, 1)

                # break if the next token is <|endoftext|>
                # if next_token_index == tokenizer.eos_token_id:
                    # break

                # append sampled token index to the running sequence
                tokens = torch.cat((tokens, next_token_index), dim=1) # (B, T+1)

        return input_tokens

# Optimizing Implementation For SPEEED

GPUS are really fast these days, and we wish to fully utilize the power of them with our code

Which really down to the heart just asks 1 question, how do we make matrix multiplication faster with our GPU not waiting around?

Let's test how we can speed things up training the toy dataset of Tiny ShakeSphere



In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2024-07-16 14:13:34--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2024-07-16 14:13:35 (34.4 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [ ]:
import tiktoken
import time


class DataLoaderSimple:

    def __init__(self, batch_size, seq_len):
        self.batch_size = batch_size
        self.seq_len = seq_len

        with open("input.txt", "r") as f:
            text = f.read()

        tokenizer = tiktoken.get_encoding("gpt2")
        tokens = tokenizer.encode(text)
        self.data = torch.tensor(tokens)

    def random_batch(self):
        B, T = self.batch_size, self.seq_len
        index = torch.randint(0, len(self.data)-B*T, (1,)).item()
        data = self.data[index:index+B*T+1]
        inputs = data[:-1].view(B, T)
        targets = data[1:].view(B, T)
        return inputs, targets


def mini_train(model, optimizer, loss_fn, train_dataloader, steps):

    model.train()

    for step in range(steps):

        t0 = time.time()
        inputs, targets = train_dataloader.random_batch()
        logits = model(inputs)

        B,T,C = logits.shape
        loss = loss_fn(logits.contiguous().view(B*T,C), targets.contiguous().view(B*T))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        torch.cuda.synchronize()
        t1 = time.time()

        diff = (t1-t0)*1000 # time diff in miliseconds

        print(f"step {step}: loss {loss.item():.4f}, time {diff:.2f}ms")

In [ ]:
# cuda stuff
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.set_default_device(device)

# hyperparameters for speed testing in mini train
steps = 10
learning_rate = 1e-3
batch_size = 4
seq_len = 1024

In [ ]:
tiny_dataloader = DataLoaderSimple(batch_size=batch_size, seq_len=seq_len)

model = Our_GPT2(Our_GPT2Config)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
loss_fn = torch.nn.CrossEntropyLoss()

# Test With Default Implemntation

In [ ]:
mini_train(model, optimizer, loss_fn, tiny_dataloader, steps)

step 0: loss 10.9812, time 1351.98ms
step 1: loss 9.7449, time 640.46ms
step 2: loss 8.7455, time 650.91ms
step 3: loss 8.0713, time 644.91ms
step 4: loss 7.5311, time 646.83ms
step 5: loss 7.2596, time 647.57ms
step 6: loss 7.0792, time 643.75ms
step 7: loss 6.5135, time 645.37ms
step 8: loss 6.8919, time 648.60ms
step 9: loss 6.6413, time 645.31ms


## Trick 1

Trick 1: mixed precision

In tensor multiplication, we don't actually need that high of a precision

In [ ]:
torch.set_float32_matmul_precision('high')

mini_train(model, optimizer, loss_fn, tiny_dataloader, steps)

step 0: loss 6.9632, time 520.66ms
step 1: loss 6.9389, time 481.81ms
step 2: loss 7.0807, time 484.60ms
step 3: loss 7.0997, time 486.67ms
step 4: loss 7.1585, time 483.14ms
step 5: loss 7.0025, time 481.94ms
step 6: loss 6.6620, time 480.85ms
step 7: loss 6.5474, time 486.27ms
step 8: loss 6.7870, time 487.11ms
step 9: loss 6.8360, time 483.38ms


## Trick 2

Trick 2: torch.autocast

No idea what it does... but it speeds things up!

In [ ]:
# with torch.autocast(device_type=device, dtype=torch.bfloat16):

In [ ]:
def mini_train(model, optimizer, loss_fn, train_dataloader, steps):

    model.train()

    for step in range(steps):

        t0 = time.time()
        inputs, targets = train_dataloader.random_batch()

        with torch.autocast(device_type=device, dtype=torch.bfloat16): # <-- the new line is here
            logits = model(inputs)
            B,T,C = logits.shape
            loss = loss_fn(logits.contiguous().view(B*T,C), targets.contiguous().view(B*T))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        torch.cuda.synchronize()
        t1 = time.time()

        diff = (t1-t0)*1000 # time diff in miliseconds

        print(f"step {step}: loss {loss.item():.4f}, time {diff:.2f}ms")

In [ ]:
mini_train(model, optimizer, loss_fn, tiny_dataloader, steps)

step 0: loss 6.6317, time 532.56ms
step 1: loss 6.3940, time 398.40ms
step 2: loss 6.4086, time 400.31ms
step 3: loss 6.7583, time 399.93ms
step 4: loss 6.4951, time 399.40ms
step 5: loss 6.1796, time 400.40ms
step 6: loss 6.9649, time 399.77ms
step 7: loss 6.4284, time 399.39ms
step 8: loss 6.6422, time 401.06ms
step 9: loss 6.6710, time 399.86ms


## Trick 3

Trick 3: torch.compile

It decreases travel time of data between gpu and cpu, basically overhead of instructions get lower

But the first step get's extremely slow...

In [ ]:
model = torch.compile(model)

mini_train(model, optimizer, loss_fn, tiny_dataloader, steps)

step 0: loss 10.9720, time 25847.32ms
step 1: loss 9.5159, time 234.46ms
step 2: loss 8.8567, time 233.67ms
step 3: loss 8.2128, time 233.35ms
step 4: loss 7.6457, time 235.98ms
step 5: loss 7.2617, time 235.19ms
step 6: loss 6.8399, time 234.47ms
step 7: loss 6.7995, time 233.27ms
step 8: loss 7.2071, time 233.79ms
step 9: loss 6.6454, time 234.19ms


## Trick 4 and Trick 5

*   trick 4: flash attention

    *   a more efficient implementation of attention



*   trick 5: less assignment, more return

    *   so instead of reassigning x again and again... just return the whole computation at once

    *   it does make code less readable, but makes things slightly faster

In [ ]:
# Flash Attention
# out = F.scaled_dot_product_attention(q, k, v, is_causal=False)

In [ ]:
from dataclasses import dataclass
from math import sqrt
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class Our_GPT2Config:
    vocab_size: int = 50257 # number of tokens: 50,000 BPE merges + 256 bytes tokens + 1 <|endoftext|> token
    embedding_size: int = 768 # embedding dimension
    seq_len: int = 1024 # max sequence length
    num_layers: int = 12 # number of layers
    num_heads: int = 12 # number of heads



# Exact Same Functionality as Char_GPT's Implementation of Attention, Just More Optimized!
class CasualSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.key = nn.Linear(config.embedding_size, config.embedding_size, bias=False)
        self.query = nn.Linear(config.embedding_size, config.embedding_size, bias=False)
        self.value = nn.Linear(config.embedding_size, config.embedding_size, bias=False)
        self.project = nn.Linear(config.embedding_size, config.embedding_size)
        self.project.flag = 1 # a "flag" for model initialization, feels like pytorch should have better implementation?

        self.num_heads = config.num_heads
        self.head_size = config.embedding_size // config.num_heads
        self.embedding_size = config.embedding_size
        self.register_buffer("mask_indexes", torch.tril(torch.ones(config.seq_len, config.seq_len)).view(1, 1, config.seq_len, config.seq_len))

    def forward(self, x):
        B, T, C = x.size() # batch_size, seq_len, embedding_size (which sometimes is called model_size)

        # Get QKV
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        # Divide into heads (man I don't know how to reshape tensors)
        q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2) # (B, num_heads, T, head_size)
        k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2) # (B, num_heads, T, head_size)
        v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2) # (B, num_heads, T, head_size)

        # Attention
        # attn = (q @ k.transpose(-2, -1)) * (1.0/sqrt(self.head_size)) # normalize by dividng by head_size
        # attn = attn.masked_fill(self.mask_indexes[:,:,:T,:T] == 0, float('-inf'))
        # attn = torch.softmax(attn, dim=-1)
        # out = attn @ v  # (B, num_heads, T, T) @ (B, num_heads, T, head_size) -> (B, num_heads, T, head_size)

        # Flash Attention
        # out = F.scaled_dot_product_attention(q, k, v, is_causal=False)


        # Get all the head outputs together
        # out = out.transpose(1, 2).contiguous().view(B, T, C)

        # Projection
        # out = self.project(out)

        return self.project(F.scaled_dot_product_attention(q, k, v, is_causal=False).transpose(1, 2).contiguous().view(B, T, C))



class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.mlp_in = nn.Linear(config.embedding_size, 4 * config.embedding_size)
        self.GELU = nn.GELU()
        self.mlp_out = nn.Linear(4 * config.embedding_size, config.embedding_size)
        self.mlp_out.flag = 1 # a "flag" for model initialization, feels like pytorch should have better implementation?

    def forward(self, x):
        # x = self.mlp_in(x)
        # x = self.GELU(x)
        # x = self.mlp_out(x)
        return self.mlp_out(self.GELU(self.mlp_in(x)))


class GPT_Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(config.embedding_size)
        self.attn = CasualSelfAttention(config)
        self.layer_norm_2 = nn.LayerNorm(config.embedding_size)
        self.mlp = MLP(config)

    # norm & add
    def forward(self, x):
        x = x + self.attn(self.layer_norm_1(x))
        x = x + self.mlp(self.layer_norm_2(x))
        return x



class Our_GPT2(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            token_embeddings = nn.Embedding(config.vocab_size, config.embedding_size),
            positional_encodings = nn.Embedding(config.seq_len, config.embedding_size),
            blocks = nn.ModuleList([GPT_Block(config) for _ in range(config.num_layers)]),
            layer_norm_final = nn.LayerNorm(config.embedding_size),
            projection = nn.Linear(config.embedding_size, config.vocab_size)
        ))

        # weight sharing scheme, so these two "share" the same tensor, and apparently this just works better than them having separate values?
        self.transformer.token_embeddings.weight = self.transformer.projection.weight

        # init weights
        self.apply(self.init_weights)


    # initialize the weights of linear layers to a normal distribution
    def init_weights(self, module):

        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'flag'):
                std *= 1/sqrt(self.config.num_layers)
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


    def forward(self, inputs):

        # inputs is of shape (B, T)
        B, T = inputs.size()

        # forward the token and position embeddings
        tok_emb = self.transformer.token_embeddings(inputs) # (B,T,C)
        pos_emb = self.transformer.positional_encodings(torch.arange(inputs.size(1), device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)

        # forward the blocks of the transformer
        for block in self.transformer.blocks:
            x = block(x)

        # forward the final layernorm and the classifier
        x = self.transformer.layer_norm_final(x)
        logits = self.transformer.projection(x) # (B, T, vocab_size)

        return logits


    def generate(self, tokens, max_seq_len):

        # inputs is (B, T) array of indices in the current context
        # but what about the conditional of '<|endoftext|>' token...?
        while tokens.size(1) <  max_seq_len:

            with torch.no_grad():

                # crop idx to the last seq_len tokens
                # context_window_tokens = tokens[:, -seq_len:]

                # get the predictions
                logits = model(tokens) # (B, T, vocab_size)

                # focus only on the last time step
                logits = logits[:, -1, :] # (B, T, vocab_size) becomes (B, vocab_size)

                # apply softmax to get probabilities
                probs = F.softmax(logits, dim=-1) # (B, vocab_size)

                # only keep the top 50 probabilities
                topk_probs, topk_indices = torch.topk(probs, 50, dim=-1) # (B, 50)

                # select a token in topk
                topk_token_index = torch.multinomial(topk_probs, num_samples=1) # (B, 1)

                # gather the correspoding index in the vocabulary
                next_token_index = torch.gather(topk_indices, dim=-1, index=topk_token_index) # (B, 1)

                # break if the next token is <|endoftext|>
                # if next_token_index == tokenizer.eos_token_id:
                    # break

                # append sampled token index to the running sequence
                tokens = torch.cat((tokens, next_token_index), dim=1) # (B, T+1)

        return input_tokens

In [ ]:
model = Our_GPT2(Our_GPT2Config)
model.to(device)
model = torch.compile(model)

mini_train(model, optimizer, loss_fn, tiny_dataloader, steps)

step 0: loss 11.0884, time 21454.49ms
step 1: loss 11.0926, time 126.93ms
step 2: loss 11.0739, time 134.30ms
step 3: loss 11.0448, time 133.88ms
step 4: loss 11.0962, time 138.76ms
step 5: loss 11.0983, time 138.02ms
step 6: loss 11.0657, time 143.42ms
step 7: loss 11.0922, time 138.82ms
step 8: loss 11.0716, time 134.99ms
step 9: loss 11.0707, time 135.13ms


## Trick 6

Trick 6: nice numbers

when numbers is divisible by 2, numbers are nice

when they are a power of 2, they are very nice

because computers, gpus all work on some power of 2... so you wanna make your "numbers" all close to some power of 2, at least make it even

In [ ]:
@dataclass
class Our_GPT2Config:
    vocab_size: int = 50304 # 50257 to this! nice number
    embedding_size: int = 768 # embedding dimension
    seq_len: int = 1024 # max sequence length
    num_layers: int = 12 # number of layers
    num_heads: int = 12 # number of heads

model = Our_GPT2(Our_GPT2Config)
model.to(device)
model = torch.compile(model)

mini_train(model, optimizer, loss_fn, tiny_dataloader, steps)

step 0: loss 11.0342, time 21814.94ms
step 1: loss 11.0559, time 118.54ms
step 2: loss 11.0081, time 134.77ms
step 3: loss 11.0117, time 134.46ms
step 4: loss 11.0439, time 126.00ms
step 5: loss 11.0307, time 124.73ms
step 6: loss 11.0380, time 126.19ms
step 7: loss 11.0540, time 125.51ms
step 8: loss 11.0041, time 124.24ms
step 9: loss 11.0097, time 125.20ms


So overall, we went from 600ms to 125ms just with

# More Tricks For Optimization (Not Speed, But Performance)

There's no need to run the code here as we can't tell how much these techniques affect learning stability

## Trick 1

Trick 1: gradient clipping

normalizes all parameters after training over an epoch...?

not exactly sure what it is, but it helps stablize training

https://stackoverflow.com/questions/54716377/how-to-do-gradient-clipping-in-pytorch

In [ ]:
# gradient clipping
# norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

In [ ]:
def mini_train(model, optimizer, loss_fn, train_dataloader, steps):

    model.train()

    for step in range(steps):

        t0 = time.time()
        inputs, targets = train_dataloader.random_batch()

        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            logits = model(inputs)
            B,T,C = logits.shape
            loss = loss_fn(logits.contiguous().view(B*T,C), targets.contiguous().view(B*T))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        # gradient clipping
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # <-- the new line is here
        optimizer.step()

        torch.cuda.synchronize()
        t1 = time.time()

        diff = (t1-t0)*1000 # time diff in miliseconds

        print(f"step {step}: loss {loss.item():.4f}, time {diff:.2f}ms")

## Trick 2

Trick 2: learning rate scheduling (cosine decay)

welp, another trick to make learning more smoother

In [ ]:
max_learning_rate = 3e-4
min_learning_rate = max_learning_rate * 0.1
warmup_steps = 5
max_steps = 25

def get_learning_rate(step):

    # linear warmup
    if step < warmup_steps:
        return max_learning_rate * (step+1) / warmup_steps

    # if it's no longer warmup, return min_learning_rate
    if step > max_steps:
        return min_learning_rate

    # in between, use cosine decay
    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    coeffient = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # starts at 1 and tends to 0 over steps
    return min_learning_rate + coeffient * (max_learning_rate - min_learning_rate)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=max_learning_rate, betas=(0.9, 0.95), eps=1e-8)

def mini_train(model, optimizer, loss_fn, train_dataloader, steps):

    model.train()

    for step in range(steps):

        t0 = time.time()
        inputs, targets = train_dataloader.random_batch()

        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            logits = model(inputs)
            B,T,C = logits.shape
            loss = loss_fn(logits.contiguous().view(B*T,C), targets.contiguous().view(B*T))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        # gradient clipping
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # <-- the new line is here

        # schedule learning rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = get_learning_rate(step)

        optimizer.step()

        torch.cuda.synchronize()
        t1 = time.time()

        diff = (t1-t0)*1000 # time diff in miliseconds

        print(f"step {step}: loss {loss.item():.4f}, time {diff:.2f}ms")

## Trick 3

Trick 3: weight decay

nope, no idea what this is, and how to implement it

copying code doesnt help me lol

## Trick 4

Trick 4: gradient accumulation

?

## Trick 5

Trick 5: DPP

this is like for utilizing multiple GPUs... not our stuff

# Datasets Used By GPT 2/3

in gpt2, it's a dataset called "WebText" which came by scraping all outbouding links from reddit which had at least 3 upvotes

GPT3 uses a lot more datasets

*   "Common Crawl (Filtered)"
*   "WebText2"
*   "Books1/2"
*   "Wikipedia"




Since they aren't open source datasets, the ones we can get our hands on to are:

*   Red Pajamas
*   Slim Pajama (Filtered version of Red Pajamas)

RedPajama is an open-source reproduction of the original LLaMA training dataset

Or other new datasets from hugging face!

*   FineWeb
*   FineWeb-EDU

We will use a sample 10 billion token from FineWeb-EDU to train, simple and small to work with

We can also evaluate the model on a dataset called "HellaSwag", which is like multiple choice testing for LLMs